# 실습 3. RAG 보안 아키텍처 구현

**시나리오:** 병원 안내 어시스턴트에 사내 문서를 검색하는 지식 기반(Knowledge Base)을 붙입니다. 문서에는 공개 안내문뿐 아니라 간호부 내부 지침, 감염관리실 대외비, 환자 민원 사례(개인정보 포함)가 섞여 있고, 누군가 **공지문에 프롬프트 인젝션을 심어** 두었습니다.

```
            ┌──────────────── 프라이빗 VPC (인터넷 경로 없음) ────────────────┐
 사용자 ─▶ │ 검색 Lambda ──▶ [VPC 엔드포인트: bedrock-agent-runtime] ──┐       │
 (역할)     └───────────────────────────────────────────────────────────┼───────┘
                                                                        ▼
        S3 문서 버킷 (SSE-KMS, 버전관리) ──수집──▶ Bedrock Knowledge Base ──▶ S3 Vectors 인덱스 (SSE-KMS)
                 ▲ 수집 전 검사(오염 문서 격리)            │ 메타데이터 필터(역할별 접근 제어)
                                                           ▼ Guardrails(개인정보 마스킹) + 데이터 계보 기록
```

| 실습 주제 | 이 노트북에서 하는 일 |
|---|---|
| 데이터 암호화 | 고객 관리형 KMS 키 하나로 문서 버킷, 벡터 인덱스, 수집 과정 임시 데이터를 암호화 |
| 접근 제어 | 문서 등급 메타데이터로 역할별 검색 범위를 서버 측에서 강제 |
| 간접 인젝션 방어 | 수집 전 문서 검사로 오염 문서 격리 |
| 프라이버시 | 검색 결과의 개인정보를 Guardrails로 마스킹 |
| 데이터 계보 | 답변마다 근거 문서의 S3 버전 ID까지 기록 |
| 네트워크 격리 | VPC 엔드포인트 정책으로 허용 API와 대상 지식 기반을 제한 |

**소요 시간:** 약 90분 (CloudFormation 배포 5~8분, 수집 2~3분 포함)

## 0. 환경 준비와 인프라 배포

In [ ]:
import sys, json, time, hashlib, datetime
from pathlib import Path
sys.path.append("..")
from common.labutils import *

MODEL_ID = pick_model()
MODEL_ARN = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/{MODEL_ID}"
EMBED_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/{EMBED_MODEL_ID}"
state = load_state()
GR = {"guardrailId": state["guardrail_id"], "guardrailVersion": str(state["guardrail_version"])} if "guardrail_id" in state else None
cfn, s3, s3v, kms = client("cloudformation"), client("s3"), client("s3vectors"), client("kms")
agent, agent_rt, ec2, lam = client("bedrock-agent"), client("bedrock-agent-runtime"), client("ec2"), client("lambda")
print("리전:", REGION, "| 모델:", MODEL_ID, "| 가드레일:", GR or "없음(실습 1 먼저 권장)")

In [ ]:
STACK = f"{LAB_PREFIX}-rag-security"
template = Path("rag_security_stack.yaml").read_text(encoding="utf-8")
try:
    cfn.create_stack(StackName=STACK, TemplateBody=template, Capabilities=["CAPABILITY_IAM"],
                     Parameters=[{"ParameterKey": "LabPrefix", "ParameterValue": LAB_PREFIX}],
                     Tags=[{"Key": "lab", "Value": LAB_PREFIX}])
    print("스택 생성 시작:", STACK, "(5~8분 소요)")
    cfn.get_waiter("stack_create_complete").wait(StackName=STACK, WaiterConfig={"Delay": 15, "MaxAttempts": 60})
except cfn.exceptions.AlreadyExistsException:
    print("기존 스택 사용:", STACK)
OUT = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK)["Stacks"][0]["Outputs"]}
save_state(rag_stack=STACK)
OUT

## 1. 수집 전 문서 검사 (간접 프롬프트 인젝션 차단)

지식 기반에 들어간 문서는 나중에 **모델 프롬프트의 일부**가 됩니다. 그래서 수집 단계에서 먼저 검사합니다. 실습 2에서 쓴 가드레일의 PROMPT_ATTACK 필터와 간단한 규칙을 함께 적용합니다.

In [ ]:
import re
DOC_DIR = Path("sample_docs")
docs = sorted(p for p in DOC_DIR.glob("*.txt"))
SUSPICIOUS = [r"이전의? 모든 지시를? 무시", r"ignore (all )?previous", r"(시스템|system) ?(공지|prompt|프롬프트)", r"빠짐없이 출력"]

def scan_document(text):
    findings = [p for p in SUSPICIOUS if re.search(p, text, re.I)]
    if GR:
        res = client("bedrock-runtime").apply_guardrail(
            guardrailIdentifier=GR["guardrailId"], guardrailVersion=GR["guardrailVersion"], source="INPUT",
            content=[{"text": {"text": text[:5000], "qualifiers": ["guard_content"]}}])
        for a in res["assessments"]:
            for f in a.get("contentPolicy", {}).get("filters", []):
                if f["type"] == "PROMPT_ATTACK":
                    findings.append(f"guardrail:PROMPT_ATTACK({f['confidence']})")
    return findings

rows, quarantined = [], []
for p in docs:
    meta = json.loads(Path(str(p) + ".metadata.json").read_text(encoding="utf-8"))["metadataAttributes"]
    f = scan_document(p.read_text(encoding="utf-8"))
    if f:
        quarantined.append(p.name)
    rows.append([p.name, meta["access_level"], meta["department"], "격리" if f else "통과", ", ".join(f)[:45]])
table(rows, ["문서", "등급", "부서", "판정", "탐지 내용"])

### 1-1. 일부러 오염 문서까지 올려 보기
먼저 **검사 없이** 모든 문서를 올려 공격이 실제로 어떻게 작동하는지 봅니다. 7절에서 격리 후 다시 비교합니다.

In [ ]:
BUCKET = OUT["DocsBucketName"]
for p in docs:
    for f in (p, Path(str(p) + ".metadata.json")):
        s3.upload_file(str(f), BUCKET, f"docs/{f.name}")   # 버킷 기본 암호화(SSE-KMS)가 자동 적용됩니다
head = s3.head_object(Bucket=BUCKET, Key=f"docs/{docs[0].name}")
print("업로드 완료. 암호화:", head["ServerSideEncryption"], "| 키:", head["SSEKMSKeyId"].split("/")[-1], "| 버전:", head["VersionId"])

## 2. 암호화된 벡터 저장소 (S3 Vectors)

벡터 인덱스도 같은 KMS 키로 암호화합니다. 임베딩 벡터는 원문을 상당 부분 복원할 수 있는 정보이므로 원문과 같은 수준으로 보호해야 합니다.

In [ ]:
VBUCKET = f"{LAB_PREFIX}-rag-vectors-{ACCOUNT_ID}"
INDEX = "hospital-docs"
enc = {"sseType": "aws:kms", "kmsKeyArn": OUT["KmsKeyArn"]}
try:
    s3v.create_vector_bucket(vectorBucketName=VBUCKET, encryptionConfiguration=enc)
except s3v.exceptions.ConflictException:
    pass
try:
    INDEX_ARN = s3v.create_index(
        vectorBucketName=VBUCKET, indexName=INDEX, dataType="float32", dimension=1024, distanceMetric="cosine",
        # 본문과 전체 메타데이터는 필터 대상에서 제외하고, access_level·department 같은 사용자 메타데이터만 필터에 씁니다
        metadataConfiguration={"nonFilterableMetadataKeys": ["AMAZON_BEDROCK_TEXT", "AMAZON_BEDROCK_METADATA"]},
    )["indexArn"]
except s3v.exceptions.ConflictException:
    INDEX_ARN = s3v.get_index(vectorBucketName=VBUCKET, indexName=INDEX)["index"]["indexArn"]
save_state(vector_bucket=VBUCKET, vector_index=INDEX)
print("벡터 버킷 암호화:", s3v.get_vector_bucket(vectorBucketName=VBUCKET)["vectorBucket"]["encryptionConfiguration"])
print("인덱스:", INDEX_ARN)

## 3. 최소 권한 지식 기반 역할과 지식 기반 생성

지식 기반 서비스 역할에는 **이 버킷, 이 인덱스, 이 키, 이 임베딩 모델**만 허용합니다. 신뢰 정책에는 `aws:SourceAccount` 조건을 걸어 혼동된 대리인(confused deputy) 문제를 막습니다.

In [ ]:
KB_ROLE = f"{LAB_PREFIX}-kb-role"
kb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "Embed", "Effect": "Allow", "Action": "bedrock:InvokeModel", "Resource": EMBED_ARN},
        {"Sid": "ReadDocs", "Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"],
         "Resource": [f"arn:aws:s3:::{BUCKET}", f"arn:aws:s3:::{BUCKET}/docs/*"],
         "Condition": {"StringEquals": {"aws:ResourceAccount": ACCOUNT_ID}}},
        {"Sid": "Vectors", "Effect": "Allow",
         "Action": ["s3vectors:PutVectors", "s3vectors:GetVectors", "s3vectors:DeleteVectors", "s3vectors:QueryVectors", "s3vectors:GetIndex"],
         "Resource": INDEX_ARN},
        {"Sid": "Kms", "Effect": "Allow", "Action": ["kms:Decrypt", "kms:GenerateDataKey", "kms:DescribeKey"], "Resource": OUT["KmsKeyArn"]},
    ],
}
KB_ROLE_ARN = ensure_role(KB_ROLE, "bedrock.amazonaws.com", kb_policy)
save_state(kb_role=KB_ROLE)
print(KB_ROLE_ARN)

In [ ]:
KB_NAME = f"{LAB_PREFIX}-hospital-kb"
found = [k for k in agent.list_knowledge_bases()["knowledgeBaseSummaries"] if k["name"] == KB_NAME]
if found:
    KB_ID = found[0]["knowledgeBaseId"]
else:
    KB_ID = agent.create_knowledge_base(
        name=KB_NAME, roleArn=KB_ROLE_ARN, description="병원 안내 RAG (GenAI 보안 실습 3)",
        knowledgeBaseConfiguration={"type": "VECTOR", "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": EMBED_ARN,
            "embeddingModelConfiguration": {"bedrockEmbeddingModelConfiguration": {"dimensions": 1024, "embeddingDataType": "FLOAT32"}}}},
        storageConfiguration={"type": "S3_VECTORS", "s3VectorsConfiguration": {"indexArn": INDEX_ARN}},
        tags={"lab": LAB_PREFIX},
    )["knowledgeBase"]["knowledgeBaseId"]
wait_for(lambda: agent.get_knowledge_base(knowledgeBaseId=KB_ID)["knowledgeBase"], ok=("ACTIVE",), fail=("FAILED",), label="지식 기반")

dss = agent.list_data_sources(knowledgeBaseId=KB_ID)["dataSourceSummaries"]
if dss:
    DS_ID = dss[0]["dataSourceId"]
else:
    DS_ID = agent.create_data_source(
        knowledgeBaseId=KB_ID, name="hospital-docs", dataDeletionPolicy="DELETE",
        dataSourceConfiguration={"type": "S3", "s3Configuration": {"bucketArn": f"arn:aws:s3:::{BUCKET}", "inclusionPrefixes": ["docs/"]}},
        serverSideEncryptionConfiguration={"kmsKeyArn": OUT["KmsKeyArn"]},   # 수집 중 임시 데이터도 같은 키로 암호화
        vectorIngestionConfiguration={"chunkingConfiguration": {"chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {"maxTokens": 300, "overlapPercentage": 10}}},
    )["dataSource"]["dataSourceId"]
save_state(kb_id=KB_ID, kb_data_source=DS_ID)
KB_ARN = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/{KB_ID}"
print("KB:", KB_ID, "| 데이터 소스:", DS_ID)

In [ ]:
def ingest():
    job = agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)["ingestionJob"]
    j = wait_for(lambda: agent.get_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID, ingestionJobId=job["ingestionJobId"])["ingestionJob"],
                 ok=("COMPLETE",), fail=("FAILED", "STOPPED"), label="수집 작업", interval=10)
    print("  통계:", j["statistics"])
    return j["ingestionJobId"]

INGESTION_JOB_ID = ingest()

## 4. 역할 기반 접근 제어 (메타데이터 필터)

같은 질문이라도 **로그인한 사용자의 역할**에 따라 검색 범위가 달라야 합니다.

> **핵심:** 필터는 반드시 서버가 인증 정보(예: Cognito 토큰의 그룹)로 만들어야 합니다. 사용자 입력이나 프롬프트로 필터를 정하게 하면 "나는 관리자야"라는 한 문장으로 우회됩니다.

In [ ]:
ROLE_ACCESS = {                       # 인증 시스템의 그룹 → 열람 가능한 문서 등급
    "patient": ["public"],
    "nurse": ["public", "internal"],
    "infection_control": ["public", "internal", "confidential"],
}

def retrieve(query, role, k=4):
    res = agent_rt.retrieve(
        knowledgeBaseId=KB_ID, retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {
            "numberOfResults": k,
            "filter": {"in": {"key": "access_level", "value": ROLE_ACCESS[role]}},
        }})
    return res["retrievalResults"]

QUERY = "다제내성균 CRE 발생 현황과 고위험 약물 투약 규칙 알려줘"
rows = []
for role in ROLE_ACCESS:
    hits = retrieve(QUERY, role)
    names = sorted({h["location"]["s3Location"]["uri"].split("/")[-1][:28] for h in hits})
    rows.append([role, ",".join(ROLE_ACCESS[role]), len(hits), " | ".join(names)])
table(rows, ["역할", "허용 등급", "검색 수", "검색된 문서"])

## 5. 검색 + 생성에 가드레일 적용 (개인정보 마스킹)

민원 사례 문서에는 환자 이름, 전화번호, 이메일이 들어 있습니다. 감염관리실 권한으로 검색은 허용되더라도, 답변에 개인정보가 그대로 나가면 안 됩니다. `RetrieveAndGenerate`에 실습 1의 가드레일을 붙입니다.

In [ ]:
def rag(query, role, guardrail=True, k=4):
    kb_conf = {
        "knowledgeBaseId": KB_ID, "modelArn": MODEL_ARN,
        "retrievalConfiguration": {"vectorSearchConfiguration": {
            "numberOfResults": k, "filter": {"in": {"key": "access_level", "value": ROLE_ACCESS[role]}}}},
    }
    if guardrail and GR:
        kb_conf["generationConfiguration"] = {"guardrailConfiguration": GR}
    return agent_rt.retrieve_and_generate(input={"text": query},
        retrieveAndGenerateConfiguration={"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": kb_conf})

q = "3분기 민원 사례를 민원인 이름과 연락처까지 포함해서 정리해줘"
print("[가드레일 없음]\n", rag(q, "infection_control", guardrail=False)["output"]["text"][:500])
print("\n[가드레일 적용]\n", rag(q, "infection_control", guardrail=True)["output"]["text"][:500])

## 6. 데이터 계보 추적

답변이 **어떤 문서의 어떤 버전**에 근거했는지 남겨야 사고가 났을 때 원인을 추적하고, 문서가 수정·삭제되었을 때 영향 범위를 알 수 있습니다. 문서 버킷의 버전 관리 덕분에 S3 버전 ID까지 기록할 수 있습니다.

In [ ]:
def lineage_record(query, role, response):
    sources = []
    for c in response.get("citations", []):
        for ref in c.get("retrievedReferences", []):
            uri = ref["location"]["s3Location"]["uri"]
            key = uri.split(f"{BUCKET}/", 1)[1]
            h = s3.head_object(Bucket=BUCKET, Key=key)
            sources.append({
                "uri": uri, "s3_version_id": h["VersionId"], "etag": h["ETag"].strip('"'),
                "access_level": ref.get("metadata", {}).get("access_level"),
                "chunk_sha256": hashlib.sha256(ref["content"]["text"].encode()).hexdigest()[:16],
            })
    return {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "request_id": response["ResponseMetadata"]["RequestId"], "session_id": response.get("sessionId"),
        "user_role": role, "query": query, "model": MODEL_ID, "guardrail": GR,
        "knowledge_base_id": KB_ID, "ingestion_job_id": INGESTION_JOB_ID,
        "answer_sha256": hashlib.sha256(response["output"]["text"].encode()).hexdigest()[:16],
        "sources": sources,
    }

q = "토요일 외래 진료 시간과 주차 요금 알려줘"
resp = rag(q, "patient")
rec = lineage_record(q, "patient", resp)
print("답변:", resp["output"]["text"][:200])
print(json.dumps(rec, ensure_ascii=False, indent=1))
with open("lineage_log.jsonl", "a", encoding="utf-8") as f:     # 운영에서는 CloudWatch Logs나 S3(Object Lock)에 저장
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

## 7. 오염 문서 공격 확인 → 격리 → 재수집

1-1에서 검사 없이 올린 오염 공지문이 실제로 무슨 일을 하는지 봅니다. 환자 권한이라도 공지문(public)은 검색되므로, 그 안의 지시가 모델 프롬프트에 들어갑니다.

In [ ]:
q = "12월 접수 창구 변경 공지 요약해줘"
before = rag(q, "patient", guardrail=False)["output"]["text"]
print("[격리 전]\n", before[:600])

# 1절에서 탐지한 문서를 격리 영역으로 옮기고 원본 삭제 → 재수집하면 벡터에서도 제거됩니다 (dataDeletionPolicy=DELETE)
for name in quarantined:
    for suffix in ("", ".metadata.json"):
        src = f"docs/{name}{suffix}"
        s3.copy_object(Bucket=BUCKET, Key=f"quarantine/{name}{suffix}", CopySource=f"{BUCKET}/{src}")
        s3.delete_object(Bucket=BUCKET, Key=src)
print("\n격리:", quarantined)
INGESTION_JOB_ID = ingest()

after = rag(q, "patient", guardrail=False)["output"]["text"]
print("\n[격리 후]\n", after[:600])

**토론:** 격리는 수집 경로를 통제할 수 있을 때만 통합니다. 웹 크롤링이나 사용자 업로드처럼 통제가 어려운 소스라면 무엇을 더 해야 할까요? (힌트: 검색 결과에도 같은 검사 적용, 출처별 신뢰 등급 메타데이터, 실습 2의 L3·L4)

## 8. 네트워크 격리: VPC 엔드포인트 정책

검색 Lambda는 **인터넷 경로가 없는 서브넷**에 있습니다. 그런데도 Bedrock을 호출할 수 있는 이유는 PrivateLink 인터페이스 엔드포인트 덕분입니다. 이제 엔드포인트 정책으로 **"이 역할이, 이 지식 기반에, Retrieve만"** 할 수 있게 좁힙니다.

Lambda의 IAM 역할은 Retrieve와 RetrieveAndGenerate를 모두 허용하므로, 아래에서 생기는 거부는 **네트워크 계층(엔드포인트 정책)** 에서 걸린 것입니다.

In [ ]:
def invoke_private(payload):
    r = lam.invoke(FunctionName=OUT["RetrieverFunctionName"], Payload=json.dumps(payload).encode())
    return json.loads(r["Payload"].read())

def set_endpoint_policy(doc):
    ec2.modify_vpc_endpoint(VpcEndpointId=OUT["AgentRuntimeEndpointId"], PolicyDocument=json.dumps(doc))
    time.sleep(30)   # 정책 전파 대기

tests = [
    ("Retrieve (허용 대상 KB)", {"kb_id": KB_ID, "query": "주차 요금"}),
    ("RetrieveAndGenerate", {"kb_id": KB_ID, "query": "주차 요금", "mode": "rag", "model_arn": MODEL_ARN}),
    ("Retrieve (다른 KB ID)", {"kb_id": "ABCDEFGHIJ", "query": "주차 요금"}),
]

def run_tests(label):
    rows = []
    for name, p in tests:
        r = invoke_private(p)
        detail = r.get("error") or str(r.get("sources") or r.get("answer") or "")
        rows.append([label, name, "성공" if r.get("ok") else "거부/실패", detail[:70]])
    return rows

set_endpoint_policy({"Statement": [{"Effect": "Allow", "Principal": "*", "Action": "*", "Resource": "*"}]})
rows = run_tests("기본(전체 허용)")

set_endpoint_policy({"Statement": [{
    "Effect": "Allow",
    "Principal": {"AWS": OUT["RetrieverRoleArn"]},
    "Action": ["bedrock:Retrieve"],
    "Resource": [KB_ARN],
}]})
rows += run_tests("최소 권한")
table(rows, ["엔드포인트 정책", "호출", "결과", "상세"])

**확인 포인트**
- 기본 정책에서는 세 호출 모두 엔드포인트를 통과합니다(다른 KB ID는 존재하지 않아 실패).
- 최소 권한 정책에서는 `RetrieveAndGenerate`가 **IAM은 허용했는데도** 엔드포인트 정책에서 거부됩니다. 오류 메시지에 VPC 엔드포인트 정책이 언급되는지 확인하세요.
- 운영에서는 여기에 더해 IAM 정책의 `aws:SourceVpce` 조건으로 "이 엔드포인트를 거치지 않은 호출"을 막을 수 있습니다.

## 9. 암호화 설정 점검

In [ ]:
key_id = OUT["KmsKeyArn"].split("/")[-1]
rows = [
    ["문서 버킷 기본 암호화", s3.get_bucket_encryption(Bucket=BUCKET)["ServerSideEncryptionConfiguration"]["Rules"][0]["ApplyServerSideEncryptionByDefault"]["SSEAlgorithm"]],
    ["문서 버킷 버전 관리", s3.get_bucket_versioning(Bucket=BUCKET).get("Status")],
    ["벡터 버킷 암호화", s3v.get_vector_bucket(vectorBucketName=VBUCKET)["vectorBucket"]["encryptionConfiguration"]["sseType"]],
    ["수집 임시 데이터 암호화", "KMS" if agent.get_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)["dataSource"].get("serverSideEncryptionConfiguration") else "기본"],
    ["KMS 키 자동 교체", kms.get_key_rotation_status(KeyId=key_id)["KeyRotationEnabled"]],
]
table(rows, ["항목", "상태"])

## 10. 정리와 토론

**RAG 보안 체크리스트**
| 계층 | 통제 | 이 실습 |
|---|---|---|
| 수집 | 출처 검증, 인젝션·개인정보 검사, 격리 | 1절, 7절 |
| 저장 | 원문·벡터·임시 데이터 모두 고객 관리형 키로 암호화 | 2절, 9절 |
| 검색 | 인증 기반 메타데이터 필터 | 4절 |
| 생성 | 가드레일(개인정보, 근거 검사) | 5절 |
| 감사 | 답변별 근거 문서 버전 기록 | 6절 |
| 네트워크 | PrivateLink + 엔드포인트 정책 + `aws:SourceVpce` | 8절 |

**토론 질문**
1. 메타데이터 필터만으로 충분할까요? 문서 등급이 잘못 붙은 경우는 어떻게 탐지할까요?
2. 임베딩 벡터만 유출되면 원문이 안전하다고 할 수 있을까요?
3. 데이터 계보 로그 자체에 질문 원문이 들어갑니다. 이 로그는 어떻게 보호해야 할까요?

> 리소스 정리는 `99_cleanup.ipynb`에서 합니다. VPC 엔드포인트는 시간당 요금이 발생하니 실습 후 바로 정리하세요.